# 02b — Moondream-Assisted Training Label Review

This notebook uses Moondream as a **human-review assistant**, not an automatic labeler. It processes only ranked training-set candidates from notebook 02. It never reads the competition test images.

The pipeline uses two passes:

1. neutral factual description without showing the original label or OOF prediction;
2. taxonomy-aware suggestion using `configs/moondream_labeling_rubric.json`.

A human reviewer must still complete the `human_*` columns before any decision is merged into the clean-dataset pipeline.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'leaderboard_pipeline':
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = REPO_ROOT / 'BDC2026'
CANDIDATE_PATH = REPO_ROOT / 'leaderboard_pipeline_outputs/02_label_review/ranked_review_queue.csv'
RUBRIC_PATH = REPO_ROOT / 'configs/moondream_labeling_rubric.json'
OUTPUT_DIR = REPO_ROOT / 'leaderboard_pipeline_outputs/02b_moondream_review'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('CANDIDATE_PATH:', CANDIDATE_PATH)
print('RUBRIC_PATH:', RUBRIC_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)

## Install the optional runtime

Run this once inside the existing `.venv` after CUDA-compatible PyTorch is working:

```bash
uv pip install -r requirements-moondream.txt
```

The default backend uses Moondream 3.1 locally through Photon. Model weights download from Hugging Face on first use.

In [ ]:
if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(f'Run notebook 02 first: {CANDIDATE_PATH}')
if not RUBRIC_PATH.exists():
    raise FileNotFoundError(RUBRIC_PATH)

candidates = pd.read_csv(CANDIDATE_PATH)
rubric = json.loads(RUBRIC_PATH.read_text(encoding='utf-8'))

print('Candidate rows:', len(candidates))
display(candidates.get('review_tier', pd.Series(dtype=str)).value_counts().to_frame('rows'))
display(pd.DataFrame(rubric['classes']).T)
print('Rubric status:', rubric.get('status'))

## Review the rubric before inference

The default rubric is deliberately conservative. It treats packaging-versus-contents and mixed-object cases as ambiguous. Update the JSON file when the official competition taxonomy provides a clearer rule. Do not let the VLM invent the competition policy.

In [ ]:
GPU_ID = '1'  # Change after checking nvidia-smi.
MODEL_NAME = 'moondream3.1-9B-A2B'
TIERS = 'A,B,C,D'

dry_run_command = [
    sys.executable, str(REPO_ROOT / 'scripts/moondream_label_review.py'),
    '--data-root', str(DATA_ROOT),
    '--candidates', str(CANDIDATE_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--rubric', str(RUBRIC_PATH),
    '--model', MODEL_NAME,
    '--tiers', TIERS,
    '--limit', '300',
    '--dry-run',
]
print(' '.join(dry_run_command))

In [ ]:
RUN_DRY_RUN = True
if RUN_DRY_RUN:
    subprocess.run(dry_run_command, cwd=REPO_ROOT, check=True)

## Smoke test on five images

Run a small smoke test before processing hundreds of images. The script checkpoints every few images and resumes automatically unless `--overwrite` is used.

In [ ]:
RUN_SMOKE_TEST = False

smoke_output = OUTPUT_DIR / 'smoke_test'
smoke_command = [
    sys.executable, str(REPO_ROOT / 'scripts/moondream_label_review.py'),
    '--data-root', str(DATA_ROOT),
    '--candidates', str(CANDIDATE_PATH),
    '--output-dir', str(smoke_output),
    '--rubric', str(RUBRIC_PATH),
    '--backend', 'photon',
    '--model', MODEL_NAME,
    '--tiers', TIERS,
    '--limit', '5',
    '--checkpoint-every', '1',
]

if RUN_SMOKE_TEST:
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = GPU_ID
    subprocess.run(smoke_command, cwd=REPO_ROOT, env=env, check=True)
else:
    print('Set RUN_SMOKE_TEST=True to run:')
    print('CUDA_VISIBLE_DEVICES=' + GPU_ID, ' '.join(smoke_command))

## Process the ranked queue

Start with tiers A–D and at most 300 images. This focuses GPU time and human attention on the strongest evidence. The model receives neither the original label nor the OOF prediction in the factual pass.

In [ ]:
RUN_BATCH = False
LIMIT = 300

batch_command = [
    sys.executable, str(REPO_ROOT / 'scripts/moondream_label_review.py'),
    '--data-root', str(DATA_ROOT),
    '--candidates', str(CANDIDATE_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--rubric', str(RUBRIC_PATH),
    '--backend', 'photon',
    '--model', MODEL_NAME,
    '--tiers', TIERS,
    '--limit', str(LIMIT),
    '--checkpoint-every', '5',
]

if RUN_BATCH:
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = GPU_ID
    subprocess.run(batch_command, cwd=REPO_ROOT, env=env, check=True)
else:
    print('Set RUN_BATCH=True to run:')
    print('CUDA_VISIBLE_DEVICES=' + GPU_ID, ' '.join(batch_command))

In [ ]:
EVIDENCE_PATH = OUTPUT_DIR / 'moondream_review_evidence.csv'
HUMAN_QUEUE_PATH = OUTPUT_DIR / 'moondream_human_review_queue.csv'
SUMMARY_PATH = OUTPUT_DIR / 'moondream_review_summary.json'

if EVIDENCE_PATH.exists():
    evidence = pd.read_csv(EVIDENCE_PATH)
    print('Processed rows:', len(evidence))
    if SUMMARY_PATH.exists():
        display(pd.Series(json.loads(SUMMARY_PATH.read_text())).to_frame('value'))
    display(evidence['moondream_decision_parse_status'].value_counts().to_frame('rows'))
    display(evidence['moondream_recommended_action'].value_counts(dropna=False).to_frame('rows'))
else:
    evidence = pd.DataFrame()
    print('No evidence file yet. Run the smoke test or batch stage.')

In [ ]:
def show_review_grid(df, n=16, cols=4, title='Moondream review candidates'):
    sample = df.head(n).reset_index(drop=True)
    if sample.empty:
        print('No rows to display')
        return
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.2, rows * 4.4))
    axes = np.atleast_1d(axes).reshape(-1)
    for ax in axes:
        ax.axis('off')
    for ax, (_, row) in zip(axes, sample.iterrows()):
        path = Path(row['resolved_path'])
        try:
            with Image.open(path) as image:
                ax.imshow(ImageOps.exif_transpose(image).convert('RGB'))
        except Exception as exc:
            ax.text(0.5, 0.5, str(exc), ha='center', va='center')
        ax.set_title(
            f"T={row.get('label')} OOF={row.get('oof_pred')} MD={row.get('moondream_suggested_label')}\n"
            f"MD conf={row.get('moondream_confidence')} ambiguous={row.get('moondream_taxonomy_ambiguity')}\n"
            f"{str(row.get('moondream_reason', ''))[:140]}",
            fontsize=8,
        )
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

if not evidence.empty:
    strong = evidence[
        evidence['moondream_disagrees_original_and_agrees_oof'].fillna(False)
        & evidence['moondream_confidence'].astype(str).str.lower().eq('high')
        & ~evidence['moondream_taxonomy_ambiguity'].fillna(True).astype(bool)
    ].sort_values('priority_score', ascending=False)
    print('Strong human-review candidates:', len(strong))
    show_review_grid(strong, title='High-priority: Moondream and OOF agree against original label')

## Human decision stage

Open `moondream_human_review_queue.csv` and complete only these columns:

- `human_action`: `keep`, `relabel`, `exclude`, or `needs_second_review`
- `human_new_label`: required only for `relabel`
- `human_reason`: required
- `human_reviewer`: required
- `human_second_review_required`

Moondream confidence is not a calibrated probability. Do not copy its recommendation without visually checking the image and applying the written rubric.

In [ ]:
merge_command = [
    sys.executable, str(REPO_ROOT / 'scripts/merge_moondream_human_decisions.py'),
    '--base-queue', str(CANDIDATE_PATH),
    '--moondream-queue', str(HUMAN_QUEUE_PATH),
    '--output', str(REPO_ROOT / 'leaderboard_pipeline_outputs/02_label_review/ranked_review_queue_with_moondream_human_decisions.csv'),
]
print('Run only after completing the human_* columns:')
print(' '.join(merge_command))

In [ ]:
RUN_MERGE = False
if RUN_MERGE:
    subprocess.run(merge_command, cwd=REPO_ROOT, check=True)
else:
    print('Merge remains disabled until human review is complete.')

## Safeguards

- The runner refuses paths outside `BDC2026/train`.
- It records `test_images_processed = 0`.
- It saves raw responses, parsed evidence, a rubric snapshot, and timing.
- It resumes after interruption.
- The merge utility applies only completed human decisions, never Moondream recommendations.
- Keep the original dataset unchanged.
- Confirm that VLM-assisted annotation is allowed by the competition rules before using any relabeled training data.